# Question B2 (10 marks)

> In Question B1, the default settings in PyTorch Tabular were enough to perform a quick experiment. In this part of the assignment, we will try out a new model.

> In Question B1, we used the Category Embedding model. This creates a feedforward neural network in which the categorical features get learnable embeddings. In this question, we will make use of a library called `Pytorch-WideDeep`. This library makes it easy to work with multimodal deep-learning problems combining images, text, and tables. We will just be utilizing the deeptabular component of this library through the `TabMlp` network:

In [1]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.models import TabMlp, WideDeep
from pytorch_widedeep import Trainer
from pytorch_widedeep.metrics import R2Score

## 1. Divide Dataset into Train, Test

> Divide the dataset (‘hdb_price_prediction.csv’) into train and test sets by using entries from the year 2020 and before as training data, and entries from 2021 and after as the test data.

In [2]:
df = pd.read_csv('hdb_price_prediction.csv')

# Define the columns to be used
cont_cols = [
    "dist_to_nearest_stn"       , 
    "dist_to_dhoby"             ,     
    "degree_centrality"         , 
    "eigenvector_centrality"    , 
    "remaining_lease_years"     , 
    "floor_area_sqm"
]

cat_cols = [
    "month"             , 
    "town"              ,  
    "flat_model_type"   , 
    "storey_range"
]

target_cols = [
    "resale_price"
]

train   = df[df["year"] <= 2020].copy()
test    = df[df["year"] >= 2021].copy()

In [3]:
print (f"Training Data Shape: {train.shape}")
print (f"Test Data Shape: {test.shape}")

Training Data Shape: (87370, 14)
Test Data Shape: (72183, 14)


In [4]:
# Print mean and std of the target variable in the different splits
print("\nTarget Variable Statistics:")
print(f"Train - Mean: {train[target_cols[0]].mean():.2f}, Std: {train[target_cols[0]].std():.2f}")
print(f"Test - Mean: {test[target_cols[0]].mean():.2f}, Std: {test[target_cols[0]].std():.2f}")


Target Variable Statistics:
Train - Mean: 442495.91, Std: 154221.57
Test - Mean: 538502.68, Std: 169181.84


---

## 2. Model

> Refer to the documentation of Pytorch-WideDeep and perform the following tasks:
https://pytorch-widedeep.readthedocs.io/en/latest/index.html

> * Use [**TabPreprocessor**](https://pytorch-widedeep.readthedocs.io/en/latest/examples/01_preprocessors_and_utils.html#2-tabpreprocessor) to create the deeptabular component using the continuous features and the categorical features. Use this component to transform the training dataset.

> * Create the [**TabMlp**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/model_components.html#pytorch_widedeep.models.tabular.mlp.tab_mlp.TabMlp) model with 2 linear layers in the MLP, with 250 and 150 neurons respectively.

> * Create a [**Trainer**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/trainer.html#pytorch_widedeep.training.Trainer) for the training of the created TabMlp model with the mean squared error (MSE) cost function. Train the model for 80 epochs using this trainer, keeping a batch size of 64. (Note: set the *num_workers* parameter to 0.)

### 2.1 `TabPreprocessor`

`TabPreprocessor` in `pytorch_widedeep` prepare tabular data (a 2D table-like data format). Handles label encoding of categorical features (to 1, 2, ...) and normilzation of continuous features

In [5]:
# TabPreprocessor for categorical + continuous features
tab_preprocessor = TabPreprocessor(
	cat_embed_cols	= cat_cols,
	continuous_cols	= cont_cols
)

# Fit on training data and transform it
X_tab_train = tab_preprocessor.fit_transform(train)
X_tab_test  = tab_preprocessor.transform(test)

# Training target
y_train = train[target_cols[0]].values
y_test  = test[target_cols[0]].values

print("X_tab_train shape:", X_tab_train.shape)
print("y_train shape:", y_train.shape)
print("X_tab_test shape:", X_tab_test.shape)
print("y_test shape:", y_test.shape)

X_tab_train shape: (87370, 10)
y_train shape: (87370,)
X_tab_test shape: (72183, 10)
y_test shape: (72183,)


/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_widedeep/preprocessing/tab_preprocessor.py:364: UserWarning: Continuous columns will not be normalised
  warnings.warn("Continuous columns will not be normalised")


### 2.2 `TabMlp`

`TabMlp` in `pytorch_widedeep` implements a Multi-Layer Perceptron

In [6]:
# Create the TabMlp model (2 hidden layers: 250, 150)
tab_mlp = TabMlp(
    column_idx      = tab_preprocessor.column_idx,      # dict with column name to column index mapping
    cat_embed_input = tab_preprocessor.cat_embed_input, # list of tuples with categorical column index and number of unique values (for embedding)
    continuous_cols = tab_preprocessor.continuous_cols, # list of continuous column names
    mlp_hidden_dims = [250, 150],                      
)

# Wrap in WideDeep container (used by Trainer)
model = WideDeep(deeptabular=tab_mlp)

model

WideDeep(
  (deeptabular): Sequential(
    (0): TabMlp(
      (cat_embed): DiffSizeCatEmbeddings(
        (embed_layers): ModuleDict(
          (emb_layer_month): Embedding(13, 6, padding_idx=0)
          (emb_layer_town): Embedding(27, 10, padding_idx=0)
          (emb_layer_flat_model_type): Embedding(44, 13, padding_idx=0)
          (emb_layer_storey_range): Embedding(18, 8, padding_idx=0)
        )
        (embedding_dropout): Dropout(p=0.0, inplace=False)
      )
      (cont_norm): Identity()
      (encoder): MLP(
        (mlp): Sequential(
          (dense_layer_0): Sequential(
            (0): Linear(in_features=43, out_features=250, bias=True)
            (1): ReLU(inplace=True)
            (2): Dropout(p=0.1, inplace=False)
          )
          (dense_layer_1): Sequential(
            (0): Linear(in_features=250, out_features=150, bias=True)
            (1): ReLU(inplace=True)
            (2): Dropout(p=0.1, inplace=False)
          )
        )
      )
    )
    (1): Linear(i

### 2.3 `Trainer`

In [7]:
from pytorch_widedeep.callbacks import EarlyStopping

earlyStopping = EarlyStopping(
    monitor                 = "val_loss",
    patience                = 5,
    restore_best_weights    = True,
    verbose                 = 1
)

# Trainer for regression (defaults to MSE)
trainer = Trainer(
    model       = model,
    objective   = "regression",
    metrics     = [R2Score()],  # optional: R2 during training
    callbacks   = [earlyStopping],
    num_workers = 0
)

---

## 3. Train

> Report the test MSE and the test R2 value that you obtained.

In [8]:
# Train for up to 80 epochs (with early stopping)
trainer.fit(
    X_tab       = X_tab_train,
    target      = y_train,
    val_split   = 0.2,          # use 20% of training data for validation (for early stopping)
    n_epochs    = 80,
    batch_size  = 64,
)

valid: 100%|██████████| 274/274 [00:00<00:00, 279.22it/s, loss=1.56e+9, metrics={'r2': 0.9341}]


Restoring model weights from the end of the best epoch


In [9]:
from sklearn.metrics import r2_score, mean_squared_error

# Get predictions on test set
test_predictions = trainer.predict(X_tab=X_tab_test)

mse = mean_squared_error(y_test, test_predictions)
r2 = r2_score(y_test, test_predictions)

print(f"Test MSE: {mse:.2f}")
print(f"Test R2: {r2:.4f}")

predict: 100%|██████████| 1128/1128 [00:02<00:00, 426.47it/s]


Test MSE: 13819988245.74
Test R2: 0.5172
